# The homology of $L$-fuzzy simplicial complexes

Javier Perera-Lago, Alvaro Torras-Casas, Rocio Gonzalez-Diaz

In this notebook we perform the matrix computations shown in *Section 7* of the article. In the first place, we define the matrices $M_0$, $M_1$ and $M_2$. Then, we follow the proof of Proposition 6.6 to reduce them and get the change-of-basis matrices divided into blocks. Finally, we solve the equations systems needed to understand $\eta_0$ and $\eta_1$.

In [1]:
!pip install sympy
!pip install abelian

In [1]:
import sympy as sp
from abelian.linalg.factorizations import smith_normal_form

## Original matrices

In [2]:
M0 = sp.Matrix([[0,0,0,0,0]])
M0

Matrix([[0, 0, 0, 0, 0]])

In [3]:
M1 = sp.Matrix([[-1,-1,0,0,0], [1,0,-1,-1,0],[0,0,1,0,-1],[0,1,0,1,1],[0,0,0,0,0]])
M1

Matrix([
[-1, -1,  0,  0,  0],
[ 1,  0, -1, -1,  0],
[ 0,  0,  1,  0, -1],
[ 0,  1,  0,  1,  1],
[ 0,  0,  0,  0,  0]])

In [4]:
M2 = sp.Matrix([[0], [0],[1],[-1],[1]])
M2

Matrix([
[ 0],
[ 0],
[ 1],
[-1],
[ 1]])

In [5]:
M0*M1

Matrix([[0, 0, 0, 0, 0]])

In [6]:
M1*M2

Matrix([
[0],
[0],
[0],
[0],
[0]])

## Functions to reduce the matrices

In [7]:
def extend_Qr(Qr, r):
    I = sp.eye(r)
    zero_top_right = sp.zeros(r, Qr.cols)
    zero_bottom_left = sp.zeros(Qr.rows, r)
    Q = sp.BlockMatrix([
        [I, zero_top_right],
        [zero_bottom_left, Qr]
    ]).as_explicit()
    return Q
    
def extend_Dr(Dr,r):
    zero_left = sp.zeros(Dr.rows,r)
    D = sp.BlockMatrix([
        [zero_left,Dr]        
    ]).as_explicit()
    return D

In [8]:
def reduce_matrices(Mlist):
    r = 0
    t = len(Mlist)-1
    P = sp.eye(Mlist[t].cols)
    Dlist = []
    MDHlist = []
    while t>=0:
        N = Mlist[t]*P.inv()
        Nr = N[:,r:]
        Pr, Dr, Qr = smith_normal_form(Nr, compute_unimod=True)
        Q = extend_Qr(Qr,r)
        D = extend_Dr(Dr,r)
        Dlist.append(D)
        MDHlist.append(P.inv()*Q)
        r = D.rank()
        t -= 1
        P = Pr
    return Dlist[::-1], MDHlist[::-1]

## Matrix reduction

In [9]:
Mlist = [M0,M1,M2]

In [10]:
Dlist, MDHlist = reduce_matrices(Mlist)

In [11]:
Dlist

[Matrix([[0, 0, 0, 0, 0]]),
 Matrix([
 [0, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]),
 Matrix([
 [1],
 [0],
 [0],
 [0],
 [0]])]

In [12]:
MDHlist

[Matrix([
 [ 1,  0,  0, 0, 0],
 [ 0,  1,  0, 0, 0],
 [ 0,  0,  1, 0, 0],
 [-1, -1, -1, 1, 0],
 [ 0,  0,  0, 0, 1]]),
 Matrix([
 [ 0,  0,  1,  0,  1],
 [ 0, -1, -1,  0, -1],
 [ 1,  0,  0,  0,  0],
 [-1,  0,  0,  0,  1],
 [ 1,  0,  0, -1,  0]]),
 Matrix([[1]])]

## Extraction of groups U, T, R, F.

In [13]:
def groups_size(M,N):
    if M.cols != N.rows: 
        raise ValueError("Incompatible matrices")
    n_u, n_t, n_r, n_f = 0, 0, 0, 0
    for i in range(M.cols):
        a = max(abs(M.col(i)))
        if a == 0:
            b = max(abs(N.row(i)))
            n_f += (b == 0)
            n_u += (b == 1)
            n_t += (b not in (0, 1))
        else:
            n_r += 1
    return [n_u, n_t, n_r, n_f]          

In [14]:
def utrf_blocks(MDH,nlist):
    clist = [0]
    s = 0
    for x in nlist:
        s += x
        clist.append(s)
    U = MDH[:,clist[0]:clist[1]]
    T = MDH[:,clist[1]:clist[2]]
    R = MDH[:,clist[2]:clist[3]]
    F = MDH[:,clist[3]:clist[4]]
    return U, T, R, F

In [15]:
nlist0 = groups_size(Dlist[0],Dlist[1])
nlist0

[3, 0, 0, 2]

In [16]:
U0, T0, R0, F0 = utrf_blocks(MDHlist[0],nlist0)
U0, T0, R0, F0

(Matrix([
 [ 1,  0,  0],
 [ 0,  1,  0],
 [ 0,  0,  1],
 [-1, -1, -1],
 [ 0,  0,  0]]),
 Matrix(5, 0, []),
 Matrix(5, 0, []),
 Matrix([
 [0, 0],
 [0, 0],
 [0, 0],
 [1, 0],
 [0, 1]]))

In [17]:
nlist1 = groups_size(Dlist[1],Dlist[2])
nlist1

[1, 0, 3, 1]

In [18]:
U1, T1, R1, F1 = utrf_blocks(MDHlist[1],nlist1)
U1, T1, R1, F1

(Matrix([
 [ 0],
 [ 0],
 [ 1],
 [-1],
 [ 1]]),
 Matrix(5, 0, []),
 Matrix([
 [ 0,  1,  0],
 [-1, -1,  0],
 [ 0,  0,  0],
 [ 0,  0,  0],
 [ 0,  0, -1]]),
 Matrix([
 [ 1],
 [-1],
 [ 0],
 [ 1],
 [ 0]]))

## L-fuzzy value of a homology class

The system in here only contains the block U1

In [19]:
U1

Matrix([
[ 0],
[ 0],
[ 1],
[-1],
[ 1]])

In [20]:
h = sp.Matrix([[-1],[1],[0],[-1],[0]])
h

Matrix([
[-1],
[ 1],
[ 0],
[-1],
[ 0]])

System with 3rd and 5th rows:

In [21]:
sp.linsolve((U1[[2,4],:],h[[2,4],:]))

{(0,)}

System with all rows

In [22]:
sp.linsolve((U1[:,:],h[:,:]))

EmptySet

## Describing the cuts family

The system contains all the matrix $M^{\Delta,H}_0$

In [23]:
G0 = MDHlist[0]
G0

Matrix([
[ 1,  0,  0, 0, 0],
[ 0,  1,  0, 0, 0],
[ 0,  0,  1, 0, 0],
[-1, -1, -1, 1, 0],
[ 0,  0,  0, 0, 1]])

System with only 3rd row

In [24]:
G0[[2,4], :]

Matrix([
[0, 0, 1, 0, 0],
[0, 0, 0, 0, 1]])

In [25]:
G0[[2,4], :].nullspace()

[Matrix([
 [1],
 [0],
 [0],
 [0],
 [0]]),
 Matrix([
 [0],
 [1],
 [0],
 [0],
 [0]]),
 Matrix([
 [0],
 [0],
 [0],
 [1],
 [0]])]

System with 1st, 2nd and 4th rows

In [26]:
G0[[0,1,3], :]

Matrix([
[ 1,  0,  0, 0, 0],
[ 0,  1,  0, 0, 0],
[-1, -1, -1, 1, 0]])

In [27]:
G0[[0,1,3], :].nullspace()

[Matrix([
 [0],
 [0],
 [1],
 [1],
 [0]]),
 Matrix([
 [0],
 [0],
 [0],
 [0],
 [1]])]

System with all rows

In [28]:
G0.nullspace()

[]